In [ ]:
# Parameters
configfile = "config/config.yml"
input_data = (
    "results/data/checkpoints/beforefilter_intermediate_empfaenger_immunologie.pq"
)
targetpop_data = "results/data/checkpoints/targetpop.pq"
display_util = "workflow/scripts/display_util.py"
util = "workflow/scripts/util.py"
output_data = "results/data/intermediate_empfaenger_immunologie.pq"
output_model = "results/data/intermediate_empfaenger_immunologie.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt
import numpy as np

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
    collist,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    EmpfaengerID,
    drop_duplicate_columns,
    common_translate,
)

### Target Population Filtering

The patients in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
rec = data["recipient_et_id_et"].copy()
targetpop = pd.read_parquet(targetpop_data)
data = data[rec.isin(targetpop["recipient_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of recipients in the data ({rec.nunique()}) and target population ({targetpop["recipient_et_id_et"].nunique()})
            to {rec[rec.isin(targetpop["recipient_et_id_et"])].nunique()} in the processed data.
        """
    )
)
del targetpop, rec

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

In [ ]:
# this analysis is not used, as the result is non conclusive
cols = [
    "antigens_acceptable",
    "antigens_unacceptable",
    "non_cytotoxic_antibodies",
    "auto_antibodies",
    "dtt_crossmatch",
    "screening",
    "donor_freq_etkas",
    "donor_freq_het",
    "hla_phenotyping",
    "pra_unit",
    "pra_percent",
    "specificities",
    "vpra_unit",
    "vpra_percent",
]
usedcols = (
    data[cols]
    .isna()
    .apply(lambda col: np.where(col, "", col.name + "+"))
    .apply(lambda col: "".join(col), axis=1)
)
usedcols = data["result_type"] + ": " + usedcols
del cols, usedcols

### Integration of Seperated Institute Data

Only {term}`ET` data is in this file, so no data processing was necessary at this step (see [](general:ic)).

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is an existing `result_type` column which gives the type of test, which differentiates between different tests (see [](general:rf)). We kept all test types.

In [ ]:
display_long_data_doc(
    data,
    ["recipient_et_id_et"],
    "sampling_date",
    "result_type",
)

### Unit Conversions

We applied common translations and removed the unit columns. No furhter steps were necessary. (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
cols = ["pra_unit", "vpra_unit"]
assert (data.loc[:, cols].nunique() == 1).all(), "More than 1 unit!?"
data = data.drop(columns=cols)
display(
    Markdown(
        f"The columns {collist(cols)} were removed as only a single unit was used."
    )
)
del cols

Among the types of Antibody Screening tests, we observe a method called "DTT".
This is unusal, as DTT is a reagent used to inactivate IgM antibodies, not a screening method itself.
It is likely that this is CDC in reality. 

In [ ]:
f, ax = plt.subplots()
val_counts = data.query("result_type == 'Antibody Screening'")[
    "screening"
].value_counts(dropna=False)
val_counts /= val_counts.sum()
val_counts.plot.bar(ax=ax)
ax.set_xlabel("Screening Type of Antibody Screening Tests")
ax.set_ylabel("Percent of Antibody Screening Tests")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: "{:.0%}".format(y)))
display(f)

In [ ]:
f, ax = plt.subplots()
val_counts = (
    data.query("result_type == 'Antibody Screening'")
    .groupby(["screening", "dtt_crossmatch"], dropna=False)
    .size()
    .unstack(fill_value=0)
)
val_counts = val_counts.div(val_counts.sum(axis=1), axis=0)
val_counts.plot.bar(stacked=True, ax=ax)
ax.set_xlabel("Screening Type of Antibody Screening Tests")
ax.set_ylabel("Percent of Tests with this Type")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: "{:.0%}".format(y)))
display(f)

With the DTT lab screening method, most tests have `dtt_crossmatch` set to "yes", however, there are also many with "no" and the for other tests, it seems also almost balanced.

What kind of tests does each patient have?

In [ ]:
val = data.query("result_type == 'Antibody Screening'").copy()
val["recipient_et_id_et"] = val["recipient_et_id_et"].astype("category")
val["screening"] = val["screening"].astype("category")
val = val.groupby(
    ["recipient_et_id_et", "screening"], observed=False, dropna=False
).size()
# Exclude those, where both are 0
tokeep = val.groupby("recipient_et_id_et", observed=True).sum()
tokeep = tokeep[tokeep > 0].index
val = val.loc[(tokeep, slice(None))].reset_index(level=0, drop=True)
val.groupby("screening", observed=True, dropna=False).agg(
    ["min", "max", "mean", "std", "median"]
).rename(
    columns={
        "min": "Minimum Count",
        "max": "Maximum Count",
        "mean": "Average Count",
        "std": "STD of Count",
        "median": "Median Count",
    }
)

We will assume that the column `dtt_crossmatch` indicates whether DTT was used for the serum submitted. When the lab screening method is DTT, this should always be "yes", because then DTT was added to the serum for the screening test.

In [ ]:
old_dtt = data["dtt_crossmatch"].copy()
new_dtt = data["dtt_crossmatch"].where(data["screening"] != "DTT", other="yes")
# Make sure nans are accounted for
changed = (old_dtt != new_dtt) & ~(old_dtt.isna() & new_dtt.isna())
changed = changed.sum()
display(
    Markdown(
        f"Overall, this changes the DTT values for {changed} ({changed / old_dtt.notna().sum() * 100:.1f}%) samples."
    )
)
data["dtt_crossmatch"] = new_dtt
del old_dtt, new_dtt, changed

In [ ]:
f, ax = plt.subplots()
val_counts = (
    data.query("result_type == 'Antibody Screening'")
    .groupby(["screening", "dtt_crossmatch"], dropna=False)
    .size()
    .unstack(fill_value=0)
)
val_counts = val_counts.div(val_counts.sum(axis=1), axis=0)
val_counts.plot.bar(stacked=True, ax=ax)
ax.set_xlabel("Screening Type of Antibody Screening Tests")
ax.set_ylabel("Percent of Tests with this Type")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: "{:.0%}".format(y)))
display(f)

### Consolidating Columns

No consolidation was necessary. (see [](general:crc))

## Intermediate Dataset

For this longitudinal dataset we recommend the `sampling_date` column as the time axis.

In [ ]:
indcols = ["recipient_et_id_et"]
data = data.sort_index(axis=1).sort_values(["sampling_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
class EmpfaengerImmunologie(EmpfaengerID):
    antigens_acceptable: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Acceptable Antigens",
        description="Which antigens were found to be acceptable?",
    )
    antigens_unacceptable: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Inacceptable Antigens",
        description="Which antigens were found to be inacceptable?",
    )
    auto_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Auto Antibodies Detected",
        description="Were auto antibodies present?",
        isin=["positive", "negative", "not tested"],
    )
    donor_freq_etkas: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Donor Frequency (ETKAS)",
        description="What was the donor frequency as used in ETKAS?",
        ge=0,
    )
    donor_freq_het: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Donor Frequency (HET)",
        description="What was the donor frequency as used in HET?",
        ge=0,
    )
    dtt_crossmatch: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="DTT Crossmacth Testing",
        description="Was testing with DTT performed?",
        isin=["yes", "no"],
    )
    enter_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Data Entry Date",
        description="When was the data entered?",
    )
    hla_phenotyping: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HLA Phenotype Detected",
        description="Which HLA Phenotype was observed?",
    )
    non_cytotoxic_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Non-Cytotoxic Antibody Testing",
        description="Was testing for non-cytotoxic antibodies performed?",
        isin=["yes", "no"],
    )
    pra_percent: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="PRA",
        description="What were the results of the PRA test in percent?",
        le=100,
        ge=0,
    )
    result_type: Series[str] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Test Result Type",
        description="Which type of lab immunology test was performed?",
        isin=[
            "Antibody Screening",
            "HLA Typing",
            "Unacceptable Test",
            "Acceptable Test",
        ],
    )
    sampling_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Sampling date",
        description="When was the sample taken?",
    )
    screening: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Screening Method",
        description="Which lab protocol method was used to take the measurement?",
        isin=["CDC", "Elisa", "Other", "DTT", "Virtual PRA", "Luminex"],
    )
    specificities: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Antibody Specifities",
        description="Which specifities were detected during an antibody test?",
    )
    vpra_percent: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="vPRA",
        description="What were the results of the vPRA test in percent?",
        le=100,
        ge=0,
    )

    class Config:
        title = "Recipient Immunology Dataset"
        description = "Each row represents a immunologic test performed for a (potential) recipient. The data is based on the 'empfaenger_immunologie.csv' file. It contains data from the ET."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(EmpfaengerImmunologie, data)

In [ ]:
EmpfaengerImmunologie.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    EmpfaengerImmunologie.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)